In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
from pyspark.sql.functions import (
    col,
    monotonically_increasing_id,
    row_number,
    rank,
    desc,
    current_date,
    year,
    max,
    count,
    sum,
    avg
)

from pyspark.sql.window import Window

In [0]:
df_dim_movies = (
    spark.table("silver.tb_info_filmes")
    .select(
        col("id_filme"),
        col("id_imdb"),
        col("titulo"),
        col("titulo_original"),
        col("idioma_original"),
        col("data_lancamento"),
        col("ano_lancamento"),
        col("duracao"),
        col("status"),
        col("sinopse"),
        col("slogan")
    )
    .dropDuplicates(["id_filme"])
)

df_dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_movies")

In [0]:
df_dim_genres = (
    spark.table("silver.tb_generos")
    .select("genero")
    .dropDuplicates()
    .withColumn(
        "id_genero",
        monotonically_increasing_id()
    )
)

df_dim_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_genres")


df_bridge_genre = (
    spark.table("silver.tb_generos")
    .join(df_dim_genres, "genero")
    .select(
        "id_filme",
        "id_genero"
    )
    .dropDuplicates()
)

df_bridge_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.bridge_movie_genre")

print("dim_genres e bridge_movie_genre criadas!")

dim_genres e bridge_movie_genre criadas!


In [0]:
df_pessoas = spark.table("silver.tb_pessoas_empresas")


df_dim_people = (
    df_pessoas
    .filter(col("tipo") != "Produtora")
    .select("nome")
    .dropDuplicates()
    .withColumn("id_pessoa", monotonically_increasing_id())
)

df_dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_people")



df_dim_companies = (
    df_pessoas
    .filter(col("tipo") == "Produtora")
    .select(col("nome").alias("empresa"))
    .dropDuplicates()
    .withColumn("id_empresa", monotonically_increasing_id())
)

df_dim_companies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_companies")


p = df_pessoas.alias("p")
dp = df_dim_people.alias("dp")

df_bridge_people = (
    p
    .filter(col("p.tipo") != "Produtora")
    .join(
        dp,
        col("p.nome") == col("dp.nome")
    )
    .select(
        col("p.id_filme"),
        col("dp.id_pessoa"),
        col("p.tipo")
    )
    .dropDuplicates()
)

df_bridge_people.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.bridge_movie_person")


p = df_pessoas.alias("p")
dc = df_dim_companies.alias("dc")

df_bridge_company = (
    p
    .filter(col("p.tipo") == "Produtora")
    .join(
        dc,
        col("p.nome") == col("dc.empresa")
    )
    .select(
        col("p.id_filme"),
        col("dc.id_empresa")
    )
    .dropDuplicates()
)

df_bridge_company.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.bridge_movie_company")

print("Dimensões e bridges criadas!")

Dimensões e bridges criadas!


In [0]:
df_dim_reviews = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .select(
        "id_filme",
        "nome_usuario",
        "avaliacao",
        "comentario"
    )
    .withColumn(
        "id_review",
        monotonically_increasing_id()
    )
)

df_dim_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_reviews")

print("dim_reviews criada!")

dim_reviews criada!


In [0]:
df_info = spark.table("silver.tb_info_filmes")
df_metricas = spark.table("silver.tb_metricas_engajamento")
df_financeiro = spark.table("silver.tb_financeiro_filmes")

df_fact = (
    df_info
    .join(
        df_metricas,
        "id_filme",
        "left"
    )
    .join(
        df_financeiro,
        "id_filme",
        "left"
    )
    .select(
        col("id_filme"),
        col("data_lancamento"),
        col("ano_lancamento"),
        col("popularidade"),
        col("media_votos"),
        col("quantidade_votos"),
        col("avaliacao_media"),
        col("numero_votos"),
        col("orcamento_usd"),
        col("receita_usd"),
        col("lucro_usd"),
        col("margem_lucro")
    )
)

df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_movies_performance")

print("fact_movies_performance criada!")

fact_movies_performance criada!


In [0]:
%sql
SELECT
    ROUND(SUM(receita_usd), 2) AS receita_total_usd
FROM gold.fact_movies_performance;

receita_total_usd
1.76335742433E11


In [0]:
%sql
SELECT
    f.id_filme,
    d.titulo,
    f.popularidade
FROM gold.fact_movies_performance f
JOIN gold.dim_movies d
    ON f.id_filme = d.id_filme
ORDER BY f.popularidade DESC
LIMIT 5;

id_filme,titulo,popularidade
565770,blue beetle,2994.357
980489,Gran Turismo,2680.593
809905,La Fellinette,2020.0
658829,The Fear Footage 2: Curse of the Tape,2019.0
557809,wwe survivor series 2018,2018.0


In [0]:
%sql
SELECT
    g.genero,
    COUNT(DISTINCT b.id_filme) AS quantidade_filmes
FROM gold.bridge_movie_genre b
JOIN gold.dim_genres g
    ON b.id_genero = g.id_genero
GROUP BY g.genero
ORDER BY quantidade_filmes DESC;

genero,quantidade_filmes
Drama,30834
Documentary,18860
Comedy,17570
Thriller,9547
Horror,9287
Romance,7075
Action,5598
Crime,4365
Animation,4214
Tv Movie,3714


In [0]:
%sql
SELECT
    f.id_filme,
    d.titulo,
    f.receita_usd,
    RANK() OVER (ORDER BY f.receita_usd DESC) AS ranking
FROM gold.fact_movies_performance f
JOIN gold.dim_movies d
    ON f.id_filme = d.id_filme
WHERE f.receita_usd IS NOT NULL
ORDER BY ranking
LIMIT 10;

id_filme,titulo,receita_usd,ranking
299534,Avengers: Endgame,2.8E9,1
76600,Avatar: The Way of Water,2.320250281E9,2
299536,AVENGERS: INFINITY WAR,2.052415039E9,3
634649,spider-man: no way home,1.921847111E9,4
420818,The Lion King,1.663075401E9,5
361743,Top Gun: Maverick,1.488732821E9,6
346698,Barbie,1.428545028E9,7
502356,The Super Mario Bros. Movie,1.355725263E9,8
284054,Black Panther,1.349926083E9,9
181808,Star Wars: The Last Jedi,1.33269883E9,10


In [0]:
%sql
SELECT
    p.nome,
    COUNT(DISTINCT b.id_filme) AS participacoes
FROM gold.bridge_movie_person b
JOIN gold.dim_people p
    ON b.id_pessoa = p.id_pessoa
JOIN gold.dim_movies m
    ON b.id_filme = m.id_filme
WHERE b.tipo = 'Ator'
  AND m.data_lancamento >= (
      SELECT MAX(data_lancamento)
      FROM gold.dim_movies
  ) - INTERVAL 2 YEARS
GROUP BY p.nome
ORDER BY participacoes DESC
LIMIT 1;

nome,participacoes


In [0]:
%sql
SELECT
    c.empresa,
    ROUND(SUM(f.lucro_usd), 2) AS lucro_total_usd
FROM gold.bridge_movie_company b
JOIN gold.dim_companies c
    ON b.id_empresa = c.id_empresa
JOIN gold.fact_movies_performance f
    ON b.id_filme = f.id_filme
JOIN gold.dim_movies m
    ON b.id_filme = m.id_filme
WHERE m.data_lancamento >= (
    SELECT MAX(data_lancamento)
    FROM gold.dim_movies
) - INTERVAL 5 YEARS
GROUP BY c.empresa
ORDER BY lucro_total_usd DESC
LIMIT 1;

empresa,lucro_total_usd
Maumau Mia Produções,0.0


In [0]:
%sql
SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,tb_credits_and_tags,false
bronze,tb_movies_financials,false
bronze,tb_movies_info,false
bronze,tb_movies_metrics,false
bronze,tb_movies_reviews,false


In [0]:
%sql
SHOW TABLES IN silver;

database,tableName,isTemporary
silver,tb_avaliacoes_usuarios,false
silver,tb_financeiro_filmes,false
silver,tb_generos,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false
silver,tb_pessoas_empresas,false


In [0]:
%sql
SHOW TABLES IN gold;

database,tableName,isTemporary
gold,bridge_movie_company,false
gold,bridge_movie_genre,false
gold,bridge_movie_person,false
gold,dim_companies,false
gold,dim_genres,false
gold,dim_movies,false
gold,dim_people,false
gold,dim_reviews,false
gold,fact_movies_performance,false
